# 进展报告自动生成

基于项目文件（GitHubClient）调用大模型（LLM）自动生成项目进展报告。

### 调用 OpenAI GPT 大模型

相比 GitHub REST API ，OpenAI 提供的大模型相关 API 迭代速度快，且不够稳定。

**GPT-4 很难能够准确的生成 OpenAI Client 相关代码**。

因此，GitHubSentinel 项目中 LLM 相关调用代码由人类编写😁。


## Prompt 优化测试

基于 `GithubClient` 模块获取的 Repo 最新进展，先在 ChatGPT 中尝试获取可用的提示词（Prompt）方案。

-  **完整的ChatGPT 对话记录【GitHubSentinel 提示词优化】**：https://chatgpt.com/share/28524ea6-2bf3-4ebe-b7d9-9c1ba5f005d2
- 以下测试使用的 LangChain 项目文件为: `./daily_progress/langchain-ai_langchain/2024-08-18.md'`


### ChatGPT（GPT-4） 生成报告

**Langchain-AI/Langchain Daily Progress Report - 2024-08-18**

### 新增功能
1. **Langchain 模块添加**
   - 新增了Langchain Box套件及其文档加载器 (`langchain-box: add langchain box package and DocumentLoader` #25506)
   - 加入了新的社区提供者—Agentic RAG 示例 (`Community: Add Union provider - Agentic RAG example` #25509)
   - 引入了更多的异步测试标准 (`standard-tests[patch]: async variations of all tests` #25501)
   - 引入了对多种区块链的支持 (`community: add supported blockchains to Blockchain Document Loader` #25428)
   
2. **文档与API更新**
   - 更新了多个集成参考文档和Langchain版本的文档 (`docs: `integrations` reference update 9` #25511, `docs 0.3 release` #25459)
   - 增加了新的文档索引和数据加载方式的说明 (`[docs]: more indexing of document loaders` #25500)

### 主要改进
1. **测试与标准化**
   - 添加了更多嵌入标准测试 (`more embeddings standard tests` #25513)
   - 引入了JSON模式的标准测试 (`json mode standard test` #25497)
   - 新增了各种文档加载器的文档 (`[Doc] Add docs for `ZhipuAIEmbeddings`` #25467)

2. **框架和规则改进**
   - 对Langchain核心模块进行了Pydantic解析器修复 (`langchain-core: added pydantic parser fix for issue #24995` #25516)
   - 增加了B(bugbear) ruff规则以提高代码质量 (`core: Add B(bugbear) ruff rules` #25520)

3. **集成与兼容性**
   - 测试了Pydantic 2和Langchain 0.3的兼容性 (`openai[major] -- test with pydantic 2 and langchain 0.3` #25503)
   - 准备了向Pydantic 2迁移的根验证器升级 (`openai[patch]: Upgrade @root_validators in preparation for pydantic 2 migration` #25491)

### 修复问题
1. **错误修复**
   - 修正了文档中的错别字和错误消息 (`docs: Fix typo in openai llm integration notebook` #25492, `docs: fix Agent deprecation msg` #25464)
   - 解决了不同的搜索模式（向量与文本）产生不同结果的问题 (`Chroma search with vector and search with text get different result using the same embedding function` #25517)
   - 修复了使用AzureSearch vectorstore时的文档ID作为键的问题 (`community : [bugfix] Use document ids as keys in AzureSearch vectorstore` #25486)

2. **系统错误与异常处理**
   - 解决了在调用特定链时缺少输入键的错误 (`Raises ValueError: Missing some input keys: {'query'} everytime I invoke 'GraphCypherQAChain.from_llm' chain with query present as input keys` #25476)
   - 修正了未知类型 'ToolMessage' 的类型错误 (`TypeError: Got unknown type 'ToolMessage'.` #25490)

---

此简报详细总结了Langchain AI项目在2024年8月18日的最新进展，包括新增功能、主要改进和问题修复，确保团队成员了解最新的项目状态和即将到来的更新。


## 前置依赖 logger 模块

In [9]:
# src/logger.py
from loguru import logger
import sys

# Configure Loguru
logger.remove()  # Remove the default logger
logger.add(sys.stdout, level="DEBUG", format="{time} {level} {message}", colorize=True)
logger.add("logs/app.log", rotation="1 MB", level="DEBUG")

# Alias the logger for easier import
LOG = logger

# Make the logger available for import with the alias
__all__ = ["LOG"]

## LLM Class 

In [10]:
import os
from openai import OpenAI  # 导入OpenAI库用于访问GPT模型
# from logger import LOG  # 导入日志模块（演示时直接导入）

class LLM:
    def __init__(self, model="deepseek-chat"):
        # 创建一个OpenAI客户端实例
        self.client = OpenAI(
            api_key= "sk-46e3a3bf58f545beaca174a0eab590a3",  # 传入 API Key
            base_url= "https://api.deepseek.com"  # 传入 API URL
        )
        # 确定使用的模型版本
        self.model = model
        # 配置日志文件，当文件大小达到1MB时自动轮转，日志级别为DEBUG
        LOG.add("daily_progress/llm_logs.log", rotation="1 MB", level="DEBUG")

    def generate_daily_report(self, markdown_content, dry_run=False):
        # 构建一个用于生成报告的提示文本，要求生成的报告包含新增功能、主要改进和问题修复
        prompt = f"以下是项目的最新进展，根据功能合并同类项，形成一份简报，至少包含：1）新增功能；2）主要改进；3）修复问题。所有输出必须使用中文。:\n\n{markdown_content}"
        
        if dry_run:
            # 如果启用了dry_run模式，将不会调用模型，而是将提示信息保存到文件中
            LOG.info("Dry run mode enabled. Saving prompt to file.")
            with open("daily_progress/prompt.txt", "w+") as f:
                f.write(prompt)
            LOG.debug("Prompt saved to daily_progress/prompt.txt")
            return "DRY RUN"

        # 日志记录开始生成报告
        LOG.info(f"Starting report generation using model: {self.model}.")
        
        try:
            # 调用OpenAI GPT模型生成报告
            response = self.client.chat.completions.create(
                model=self.model,  # 指定使用的模型版本
                messages=[
                    {
                        "role": "system",
                        "content": "你是一个专业的项目进展分析助手。请根据用户提供的内容，整理出一份简报，要求：1）新增功能；2）主要改进；3）修复问题。所有输出必须使用中文，且语言风格简洁明了。"
                    },
                    {
                        "role": "user",
                        "content": f"请整理以下项目进展内容：\n\n{markdown_content}"
                    }
                ]
            )
            LOG.debug(f"{self.model} response: {response}")
            # 返回模型生成的内容
            return response.choices[0].message.content
        except Exception as e:
            # 如果在请求过程中出现异常，记录错误并抛出
            LOG.error("An error occurred while generating the report: {}", e)
            raise


## ReportGenerator Class

In [11]:
# src/report_generator.py

import os
from datetime import date, timedelta
# from logger import LOG  # 导入日志模块，用于记录日志信息

class ReportGenerator:
    def __init__(self, llm):
        self.llm = llm  # 初始化时接受一个LLM实例，用于后续生成报告

    def export_daily_progress(self, repo, updates):
        # 构建仓库的日志文件目录
        repo_dir = os.path.join('daily_progress', repo.replace("/", "_"))
        os.makedirs(repo_dir, exist_ok=True)  # 如果目录不存在则创建
        
        # 创建并写入日常进展的Markdown文件
        file_path = os.path.join(repo_dir, f'{date.today()}.md')
        with open(file_path, 'w') as file:
            file.write(f"# Daily Progress for {repo} ({date.today()})\n\n")
            file.write("\n## Issues\n")
            for issue in updates['issues']:
                file.write(f"- {issue['title']} #{issue['number']}\n")
            file.write("\n## Pull Requests\n")
            for pr in updates['pull_requests']:
                file.write(f"- {pr['title']} #{pr['number']}\n")
        return file_path

    def export_progress_by_date_range(self, repo, updates, days):
        # 构建目录并写入特定日期范围的进展Markdown文件
        repo_dir = os.path.join('daily_progress', repo.replace("/", "_"))
        os.makedirs(repo_dir, exist_ok=True)

        today = date.today()
        since = today - timedelta(days=days)  # 计算起始日期
        
        date_str = f"{since}_to_{today}"  # 格式化日期范围字符串
        file_path = os.path.join(repo_dir, f'{date_str}.md')
        
        with open(file_path, 'w') as file:
            file.write(f"# Progress for {repo} ({since} to {today})\n\n")
            file.write("\n## Issues Closed in the Last {days} Days\n")
            for issue in updates['issues']:
                file.write(f"- {issue['title']} #{issue['number']}\n")
            file.write("\n## Pull Requests Merged in the Last {days} Days\n")
            for pr in updates['pull_requests']:
                file.write(f"- {pr['title']} #{pr['number']}\n")
        
        LOG.info(f"Exported time-range progress to {file_path}")  # 记录导出日志
        return file_path

    def generate_daily_report(self, markdown_file_path):
        # 读取Markdown文件并使用LLM生成日报
        with open(markdown_file_path, 'r') as file:
            markdown_content = file.read()

        report = self.llm.generate_daily_report(markdown_content)  # 调用LLM生成报告

        report_file_path = os.path.splitext(markdown_file_path)[0] + "_report.md"
        with open(report_file_path, 'w+') as report_file:
            report_file.write(report)  # 写入生成的报告

        LOG.info(f"Generated report saved to {report_file_path}")  # 记录生成报告日志

    def generate_report_by_date_range(self, markdown_file_path, days):
        # 生成特定日期范围的报告，流程与日报生成类似
        with open(markdown_file_path, 'r') as file:
            markdown_content = file.read()

        report = self.llm.generate_daily_report(markdown_content)

        report_file_path = os.path.splitext(markdown_file_path)[0] + f"_report.md"
        with open(report_file_path, 'w+') as report_file:
            report_file.write(report)

        LOG.info(f"Generated report saved to {report_file_path}")  # 记录生成报告日志


## 调用 GPT-3.5-Turbo API 生成报告

In [12]:
# 实例化 LLM，并使用默认的 GPT-3.5-Turbo 模型
llm = LLM()

In [13]:
# 实例化 ReportGenerator
report_generator = ReportGenerator(llm)

In [14]:
# 生成 LangChain 项目最近一日报告
report_generator.generate_daily_report(
    markdown_file_path="daily_progress/langchain-ai_langchain/2025-01-09_to_2025-01-12.md")

2025-01-12T19:33:28.368123+0800 INFO Starting report generation using model: deepseek-chat.
2025-01-12T19:33:39.768031+0800 DEBUG deepseek-chat response: ChatCompletion(id='eea3552b-0280-49cf-a8b3-1bad02ef20e6', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='### 项目进展简报 (2025-01-09 至 2025-01-12)\n\n#### 新增功能\n1. **新增集成**：添加了Alibaba-Cloud的Lindorm作为新的集成（#29123, #29085）。\n2. **新增文档**：添加了Docling加载器的文档（#29104）。\n3. **新增支持**：支持Gemini工具调用和ChatOpenAI中的正确使用元数据（#29002）。\n4. **新增功能**：在langchain-openai中支持带有json_schema响应格式的流式传输（#29044）。\n\n#### 主要改进\n1. **文档更新**：更新了IBM WatsonxRerank的文档（#29138），并修复了文档中的导入错误（#29124）。\n2. **版本发布**：发布了多个新版本，包括langchain-openai 0.3（#29100）、xai 0.2（#29132）、pinecone 0.2.2（#29139）等。\n3. **标志更新**：将所有LANGCHAIN_标志替换为LANGSMITH_标志（#29120）。\n4. **代码优化**：优化了多个社区模块的代码，包括修复了Writer提供程序的关键错误（#27879）和解决了Confluence加载器的标签包含问题（#29089）。\n\n#### 修复问题\n1. **修复文档错误**：修复了文档中的链接错误（#29112）和标题中的额外字符（#29096）。\n2. **修复代码问题**：修复了langchain_communit

## 调用 GPT-4-Turbo API 生成报告

In [ ]:
# 实例化 LLM，并使用指定的 GPT-4-Turbo 模型
gpt_4 = LLM(model="gpt-4-turbo")

In [ ]:
# 实例化 ReportGenerator, 并使用指定的 GPT-4-Turbo 模型
rg_gpt_4 = ReportGenerator(gpt_4)

In [ ]:
# 生成 LangChain 项目最近一日报告
rg_gpt_4.generate_daily_report(
    markdown_file_path="daily_progress/langchain-ai_langchain/2024-08-18.md")

### Homework: 与ChatGPT深度对话，尝试使用 System role 提升报告质量和稳定性